# Getting started with TinyTimeMixer (TTM)

This notebooke demonstrates the usage of a pre-trained `TinyTimeMixer` model for several multivariate time series forecasting tasks. For details related to model architecture, refer to the [TTM paper](https://arxiv.org/pdf/2401.03955.pdf).

In this example, we will use a pre-trained TTM-512-96 model. That means the TTM model can take an input of 512 time points (`context_length`), and can forecast upto 96 time points (`forecast_length`) in the future. We will use the pre-trained TTM in two settings:
1. **Zero-shot**: The pre-trained TTM will be directly used to evaluate on the `test` split of the target data. Note that the TTM was NOT pre-trained on the target data.
2. **Few-shot**: The pre-trained TTM will be quickly fine-tuned on only 5% of the `train` split of the target data, and subsequently, evaluated on the `test` part of the target data.

Note: Alternatively, this notebook can be modified to try any other TTM model from a suite of TTM models. For details, visit the [Hugging Face TTM Model Repository](https://huggingface.co/ibm-granite/granite-timeseries-ttm-r2).

1. IBM Granite TTM-R1 pre-trained models can be found here: [Granite-TTM-R1 Model Card](https://huggingface.co/ibm-granite/granite-timeseries-ttm-r1)
2. IBM Granite TTM-R2 pre-trained models can be found here: [Granite-TTM-R2 Model Card](https://huggingface.co/ibm-granite/granite-timeseries-ttm-r2)
3. Research-use (non-commercial use only) TTM-R2 pre-trained models can be found here: [Research-Use-TTM-R2](https://huggingface.co/ibm-research/ttm-research-r2)

### The get_model() utility
TTM Model card offers a suite of models with varying `context_length` and `prediction_length` combinations.
In this notebook, we will utilize the TSFM `get_model()` utility that automatically selects the right model based on the given input `context_length` and `prediction_length` (and some other optional arguments) abstracting away the internal complexity. See the usage examples below in the `zeroshot_eval()` and `fewshot_finetune_eval()` functions. For more details see the [docstring](https://github.com/ibm-granite/granite-tsfm/blob/main/tsfm_public/toolkit/get_model.py) of the function definition.

## Install `tsfm` 
**[Optional for Local Run / Mandatory for Google Colab]**  
Run the below cell to install `tsfm`. Skip if already installed.

In [22]:
# # Install the tsfm library
# ! pip install "granite-tsfm[notebooks] @ git+https://github.com/ibm-granite/granite-tsfm.git@v0.3.3"

## Imports

In [23]:
import math
import os
import tempfile

import pandas as pd
import numpy as np
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from transformers import EarlyStoppingCallback, Trainer, TrainingArguments, set_seed
from transformers.integrations import INTEGRATION_TO_CALLBACK

from tsfm_public import TimeSeriesPreprocessor, TrackingCallback, count_parameters, get_datasets
from tsfm_public.toolkit.get_model import get_model
from tsfm_public.toolkit.lr_finder import optimal_lr_finder
from tsfm_public.toolkit.visualization import plot_predictions
import warnings


# Suppress all warnings
warnings.filterwarnings("ignore")

In [24]:
OUT_DIR = "ttm_finetuned_models/t"

## Finetune evaluation method

In [25]:
def fewshot_finetune_eval(
    dataset_name,
    batch_size,
    data,
    learning_rate=None,
    context_length=512,
    forecast_length=96,
    fewshot_percent=5,
    freeze_backbone=True,
    num_epochs=50,
    save_dir=OUT_DIR,
    loss="mse",
    quantile=0.5,
):
    out_dir = os.path.join(save_dir, dataset_name)

    print("-" * 20, f"Running few-shot {fewshot_percent}%", "-" * 20)

    # Data prep: Get dataset

    tsp = TimeSeriesPreprocessor(
        **column_specifiers,
        context_length=context_length,
        prediction_length=forecast_length,
        scaling=True,
        encode_categorical=False,
        scaler_type="standard",
    )

    # change head dropout to 0.7 for ett datasets
    if "ett" in dataset_name:
        finetune_forecast_model = get_model(
            TTM_MODEL_PATH,
            context_length=context_length,
            prediction_length=forecast_length,
            freq_prefix_tuning=False,
            freq=None,
            prefer_l1_loss=False,
            prefer_longer_context=True,
            # Can also provide TTM Config args
            head_dropout=0.7,
            loss=loss,
            quantile=quantile,
        )
    else:
        finetune_forecast_model = get_model(
            TTM_MODEL_PATH,
            context_length=context_length,
            prediction_length=forecast_length,
            freq_prefix_tuning=False,
            freq=None,
            prefer_l1_loss=False,
            prefer_longer_context=True,
            # Can also provide TTM Config args
            loss=loss,
            quantile=quantile,
        )

    dset_train, dset_val, dset_test = get_datasets(
        tsp,
        data,
        split_config,
        fewshot_fraction=fewshot_percent / 100,
        fewshot_location="first",
        use_frequency_token=finetune_forecast_model.config.resolution_prefix_tuning,
    )

    if freeze_backbone:
        print(
            "Number of params before freezing backbone",
            count_parameters(finetune_forecast_model),
        )

        # Freeze the backbone of the model
        for param in finetune_forecast_model.backbone.parameters():
            param.requires_grad = False

        # Count params
        print(
            "Number of params after freezing the backbone",
            count_parameters(finetune_forecast_model),
        )

    # Find optimal learning rate
    # Use with caution: Set it manually if the suggested learning rate is not suitable
    if learning_rate is None:
        learning_rate, finetune_forecast_model = optimal_lr_finder(
            finetune_forecast_model,
            dset_train,
            batch_size=batch_size,
        )
        print("OPTIMAL SUGGESTED LEARNING RATE =", learning_rate)

    print(f"Using learning rate = {learning_rate}")
    finetune_forecast_args = TrainingArguments(
        output_dir=os.path.join(out_dir, "output"),
        overwrite_output_dir=True,
        learning_rate=learning_rate,
        num_train_epochs=num_epochs,
        do_eval=True,
        eval_strategy="epoch",
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        dataloader_num_workers=8,
        report_to="none",
        save_strategy="epoch",
        logging_strategy="epoch",
        save_total_limit=1,
        logging_dir=os.path.join(out_dir, "logs"),  # Make sure to specify a logging directory
        load_best_model_at_end=True,  # Load the best model when training ends
        metric_for_best_model="eval_loss",  # Metric to monitor for early stopping
        greater_is_better=False,  # For loss
        seed=SEED,
    )

    # Create the early stopping callback
    early_stopping_callback = EarlyStoppingCallback(
        early_stopping_patience=10,  # Number of epochs with no improvement after which to stop
        early_stopping_threshold=1e-5,  # Minimum improvement required to consider as improvement
    )
    tracking_callback = TrackingCallback()

    # Optimizer and scheduler
    optimizer = AdamW(finetune_forecast_model.parameters(), lr=learning_rate)
    scheduler = OneCycleLR(
        optimizer,
        learning_rate,
        epochs=num_epochs,
        steps_per_epoch=math.ceil(len(dset_train) / (batch_size)),
    )

    finetune_forecast_trainer = Trainer(
        model=finetune_forecast_model,
        args=finetune_forecast_args,
        train_dataset=dset_train,
        eval_dataset=dset_val,
        callbacks=[early_stopping_callback, tracking_callback],
        optimizers=(optimizer, scheduler),
    )
    finetune_forecast_trainer.remove_callback(INTEGRATION_TO_CALLBACK["codecarbon"])

    # Fine tune
    finetune_forecast_trainer.train()

    # Evaluation
    print("+" * 20, f"Test MSE after few-shot {fewshot_percent}% fine-tuning", "+" * 20)

    finetune_forecast_trainer.model.loss = "mse"  # fixing metric to mse for evaluation

    fewshot_output = finetune_forecast_trainer.evaluate(dset_test)
    print(fewshot_output)
    # print("+" * 60)

    # get predictions

    predictions_dict = finetune_forecast_trainer.predict(dset_test)

    predictions_np = predictions_dict.predictions[0]

    print(predictions_np.shape)

    # get backbone embeddings (if needed for further analysis)

    backbone_embedding = predictions_dict.predictions[1]

    # print(backbone_embedding.shape)
    return dset_test, predictions_np
    # # plot
    # plot_predictions(
    #     model=finetune_forecast_trainer.model,
    #     dset=dset_test,
    #     plot_dir=os.path.join(OUT_DIR, dataset_name),
    #     plot_prefix="test_fewshot",
    #     indices=[685, 118, 902, 1984, 894, 967, 304, 57, 265, 1015],
    #     channel=0,
    # )

# Zeroshot

In [26]:
# dset_test, preds=zeroshot_eval(
#     dataset_name=TARGET_DATASET, context_length=CONTEXT_LENGTH, forecast_length=PREDICTION_LENGTH, batch_size=64
# )

In [27]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def calculate_metrics(dset_test, preds, model_name="Model",):
    """
    Calculate comprehensive evaluation metrics.
    
    Args:
        dset_test: Test dataset
        preds: Predictions array
        model_name: Name to display in output
    
    Returns:
        dict: Dictionary containing all metrics
    """
    # Extract ground truth values from dset_test
    y_true_list = []
    for i in range(len(dset_test)):
        sample = dset_test[i]
        future_values = sample['future_values']
        if hasattr(future_values, "detach"):
            future_values = future_values.detach().cpu().numpy()
        else:
            future_values = np.asarray(future_values)
        y_true_list.append(future_values)
    
    y_true = np.array(y_true_list)
    # print()
    # print(f"Predictions shape: {preds.shape}")
    # print(f"Ground truth shape: {y_true.shape}")
    
    # Validate shapes match
    if preds.shape != y_true.shape:
        raise ValueError(f"Shape mismatch! Predictions: {preds.shape}, Ground truth: {y_true.shape}")
    
    # print("\n" + "="*50)
    # print(f"{model_name} - OVERALL EVALUATION METRICS")
    # print("="*50)
    
    # Flatten for overall metric calculations
    y_true_flat = y_true.flatten()
    preds_flat = preds.flatten()
    epsilon = 1e-8
    
    # Calculate overall MSE (Mean Squared Error)
    mse = float(mean_squared_error(y_true_flat, preds_flat))
    # print(f"MSE (Mean Squared Error):        {mse:.6f}")
    
    # Calculate overall RMSE (Root Mean Squared Error)
    rmse = float(np.sqrt(mse))
    # print(f"RMSE (Root Mean Squared Error):  {rmse:.6f}")
    
    # Calculate overall MAE (Mean Absolute Error)
    mae = float(mean_absolute_error(y_true_flat, preds_flat))
    # print(f"MAE (Mean Absolute Error):       {mae:.6f}")
    
    # Calculate overall MAPE (Mean Absolute Percentage Error)
    mape = float(np.mean(np.abs((y_true_flat - preds_flat) / (y_true_flat + epsilon))) * 100)
    # print(f"MAPE (Mean Absolute % Error):    {mape:.4f}%")
    
    # Calculate overall R² Score
    r2 = float(r2_score(y_true_flat, preds_flat))
    # print(f"R² Score:                        {r2:.6f}")
    
    # Calculate overall SMAPE (Symmetric Mean Absolute Percentage Error)
    smape = float(np.mean(2.0 * np.abs(preds_flat - y_true_flat) / (np.abs(preds_flat) + np.abs(y_true_flat) + epsilon)) * 100)
    # print(f"SMAPE (Symmetric MAPE):          {smape:.4f}%")
    
    # print("="*50)
    
    # Calculate per-channel metrics
    num_samples, num_timesteps, num_channels = y_true.shape
    # print(f"\n{model_name} - PER-CHANNEL METRICS")
    # print(f"Shape: {num_samples} samples × {num_timesteps} timesteps × {num_channels} channels")
    # print("="*50)
    
    per_channel_metrics = {}
    for channel in range(num_channels):
        channel_name = target_columns[channel] if channel < len(target_columns) else f"Channel {channel}"
        
        # Extract channel data: shape is (samples, timesteps, channels)
        y_true_channel = y_true[:, :, channel].flatten()
        preds_channel = preds[:, :, channel].flatten()
        
        # Calculate metrics for this channel
        ch_mse = float(mean_squared_error(y_true_channel, preds_channel))
        ch_rmse = float(np.sqrt(ch_mse))
        ch_mae = float(mean_absolute_error(y_true_channel, preds_channel))
        ch_mape = float(np.mean(np.abs((y_true_channel - preds_channel) / (y_true_channel + epsilon))) * 100)
        ch_r2 = float(r2_score(y_true_channel, preds_channel))
        ch_smape = float(np.mean(2.0 * np.abs(preds_channel - y_true_channel) / (np.abs(preds_channel) + np.abs(y_true_channel) + epsilon)) * 100)
        
        per_channel_metrics[channel_name] = {
            'mse': ch_mse,
            'rmse': ch_rmse,
            'mae': ch_mae,
            'mape': ch_mape,
            'r2': ch_r2,
            'smape': ch_smape,
        }
        
    #     print(f"\n{channel_name}:")
    #     print(f"  MSE:   {ch_mse:.6f}")
    #     print(f"  RMSE:  {ch_rmse:.6f}")
    #     print(f"  MAE:   {ch_mae:.6f}")
    #     print(f"  MAPE:  {ch_mape:.4f}%")
    #     print(f"  R²:    {ch_r2:.6f}")
    #     print(f"  SMAPE: {ch_smape:.4f}%")
    
    # print("="*50)
    
    return {
        'overall': {
            'mse': mse,
            'rmse': rmse,
            'mae': mae,
            'mape': mape,
            'r2': r2,
            'smape': smape
        },
        'per_channel': per_channel_metrics
    }

# Calculate metrics for TTM predictions
# ttm_metrics = calculate_metrics(dset_test, preds, model_name="TTM Zero-Shot")


In [28]:
def calculate_baseline_metrics(dset, method='mean'):
    """
    Calculate baseline metrics using simple statistical methods.
    
    Args:
        dset: Dataset containing past_values and future_values
        method: 'mean' or 'median' - aggregation method for baseline
    
    Returns:
        tuple: (predictions array, metrics dictionary)
    """
    preds = []
    y_true_list = []
    
    for i in range(len(dset)):
        past_values = dset[i]['past_values']
        if hasattr(past_values, "detach"):
            past = past_values.detach().cpu().numpy()
        else:
            past = np.asarray(past_values)
        
        if method == 'mean':
            baseline_value = np.mean(past, axis=0)
        elif method == 'median':
            baseline_value = np.median(past, axis=0)
        else:
            raise ValueError(f"Unknown method: {method}. Use 'mean' or 'median'")
        
        pred = np.tile(baseline_value, (PREDICTION_LENGTH, 1))
        preds.append(pred)
        future_values = dset[i]['future_values']
        if hasattr(future_values, "detach"):
            future_values = future_values.detach().cpu().numpy()
        else:
            future_values = np.asarray(future_values)
        y_true_list.append(future_values)
    
    y_true = np.array(y_true_list)
    preds = np.array(preds)
    
    # Validate shapes match
    if preds.shape != y_true.shape:
        raise ValueError(f"Shape mismatch! Predictions: {preds.shape}, Ground truth: {y_true.shape}")
    
    # Calculate overall metrics
    y_true_flat = y_true.flatten()
    preds_flat = preds.flatten()
    epsilon = 1e-8
    
    mse = float(mean_squared_error(y_true_flat, preds_flat))
    rmse = float(np.sqrt(mse))
    mae = float(mean_absolute_error(y_true_flat, preds_flat))
    mape = float(np.mean(np.abs((y_true_flat - preds_flat) / (y_true_flat + epsilon))) * 100)
    r2 = float(r2_score(y_true_flat, preds_flat))
    smape = float(np.mean(2.0 * np.abs(preds_flat - y_true_flat) / (np.abs(preds_flat) + np.abs(y_true_flat) + epsilon)) * 100)
    
    # print("\n" + "="*50)
    # print(f"{method.upper()} Baseline - OVERALL EVALUATION METRICS")
    # print("="*50)
    # print(f"MSE (Mean Squared Error):        {mse:.6f}")
    # print(f"RMSE (Root Mean Squared Error):  {rmse:.6f}")
    # print(f"MAE (Mean Absolute Error):       {mae:.6f}")
    # print(f"MAPE (Mean Absolute % Error):    {mape:.4f}%")
    # print(f"R² Score:                        {r2:.6f}")
    # print(f"SMAPE (Symmetric MAPE):          {smape:.4f}%")
    # print("="*50)
    
    # Calculate per-channel metrics
    num_samples, num_timesteps, num_channels = y_true.shape
    print(f"\n{method.upper()} Baseline - PER-CHANNEL METRICS")
    print(f"Shape: {num_samples} samples × {num_timesteps} timesteps × {num_channels} channels")
    print("="*50)
    
    per_channel_metrics = {}
    for channel in range(num_channels):
        channel_name = target_columns[channel] if channel < len(target_columns) else f"Channel {channel}"
        
        # Extract channel data
        y_true_channel = y_true[:, :, channel].flatten()
        preds_channel = preds[:, :, channel].flatten()
        
        # Calculate metrics for this channel
        ch_mse = float(mean_squared_error(y_true_channel, preds_channel))
        ch_rmse = float(np.sqrt(ch_mse))
        ch_mae = float(mean_absolute_error(y_true_channel, preds_channel))
        ch_mape = float(np.mean(np.abs((y_true_channel - preds_channel) / (y_true_channel + epsilon))) * 100)
        ch_r2 = float(r2_score(y_true_channel, preds_channel))
        ch_smape = float(np.mean(2.0 * np.abs(preds_channel - y_true_channel) / (np.abs(preds_channel) + np.abs(y_true_channel) + epsilon)) * 100)
        
        per_channel_metrics[channel_name] = {
            'mse': ch_mse,
            'rmse': ch_rmse,
            'mae': ch_mae,
            'mape': ch_mape,
            'r2': ch_r2,
            'smape': ch_smape,
        }
        
    #     print(f"\n{channel_name}:")
    #     print(f"  MSE:   {ch_mse:.6f}")
    #     print(f"  RMSE:  {ch_rmse:.6f}")
    #     print(f"  MAE:   {ch_mae:.6f}")
    #     print(f"  MAPE:  {ch_mape:.4f}%")
    #     print(f"  R²:    {ch_r2:.6f}")
    #     print(f"  SMAPE: {ch_smape:.4f}%")
    
    # print("="*50)
    
    metrics = {
        'overall': {
            'mse': mse,
            'rmse': rmse,
            'mae': mae,
            'mape': mape,
            'r2': r2,
            'smape': smape
        },
        'per_channel': per_channel_metrics
    }
    
    return preds, metrics


# Calculate baseline metrics
# mean_baseline_preds, mean_baseline_metrics = calculate_baseline_metrics(dset_test, method='mean')
# median_baseline_preds, median_baseline_metrics = calculate_baseline_metrics(dset_test, method='median')

In [29]:
SEED = 42
set_seed(SEED)

# TTM Model path. The default model path is Granite-R2. Below, you can choose other TTM releases.
TTM_MODEL_PATH = "ibm-granite/granite-timeseries-ttm-r2"
# TTM_MODEL_PATH = "ibm-granite/granite-timeseries-ttm-r1"
# TTM_MODEL_PATH = "ibm-research/ttm-research-r2"

# Context length, Or Length of the history.
# Currently supported values are: 512/1024/1536 for Granite-TTM-R2 and Research-Use-TTM-R2, and 512/1024 for Granite-TTM-R1
CONTEXT_LENGTH = 512
#1  week or 2 weeks, predict for next 2 days
# Granite-TTM-R2 supports forecast length upto 720 and Granite-TTM-R1 supports forecast length upto 96
# Arima? Rolling average, Rolling median 
PREDICTION_LENGTH = 96
OUT_DIR = "ttm_finetuned_models/"

In [30]:
import json
from datetime import datetime
from tqdm import tqdm

folder = r"/home/rishi/ML Projects/Air Pollution/temp"#CPCB/sites_imputed"
files = os.listdir(folder)  # Fixed - get all files in the folder
timestamp_column = "Timestamp"
id_columns = []  # mention the ids that uniquely identify a time-series.

target_columns = [
 'PM2.5 (µg/m³)',
 'PM10 (µg/m³)',
 'NO2 (µg/m³)',
 'SO2 (µg/m³)',
 'CO (mg/m³)',
 'Ozone (µg/m³)',
]

split_config = {
    "train": 0.6,
    "test": 0.2,
}

column_specifiers = {
    "timestamp_column": timestamp_column,
    "id_columns": id_columns,
    "target_columns": target_columns,
    "control_columns": [],
}

# Create output directory for results
results_dir = "ttm_benchmarking_results3"
os.makedirs(results_dir, exist_ok=True)

# Store all results
all_results = []

for file in tqdm(files):
    if not file.endswith('.csv'):
        continue
        
    print(f"\n{'='*60}")
    print(f"Processing: {file}")
    print(f"{'='*60}")
    
    try:
        data = pd.read_csv(
            os.path.join(folder, file),
            parse_dates=[timestamp_column],
        ).copy()
        
        site_name = file.replace('.csv', '')
        
        # Run zero-shot evaluation
        dset_test, preds = fewshot_finetune_eval(
            dataset_name=site_name, 
            data=data,
            context_length=CONTEXT_LENGTH, 
            forecast_length=PREDICTION_LENGTH, 
            batch_size=64
        )
        
        # Calculate TTM metrics
        ttm_metrics = calculate_metrics(dset_test, preds, model_name=f"TTM - {site_name}")
        
        # Calculate baseline metrics (mean)
        mean_baseline_preds, mean_baseline_metrics = calculate_baseline_metrics(dset=dset_test, method='mean')
        
        # Calculate baseline metrics (median)
        median_baseline_preds, median_baseline_metrics = calculate_baseline_metrics(dset=dset_test, method='median')
        
        # Store results
        result = {
            'site': site_name,
            'file': file,
            'timestamp': datetime.now().isoformat(),
            'context_length': CONTEXT_LENGTH,
            'prediction_length': PREDICTION_LENGTH,
            'ttm_metrics': ttm_metrics,
            'mean_baseline_metrics': mean_baseline_metrics,
            'median_baseline_metrics': median_baseline_metrics
        }
        all_results.append(result)
        
        # Save individual site results
        site_result_file = os.path.join(results_dir, f"{site_name}_metrics.json")
        with open(site_result_file, 'w') as f:
            json.dump(result, f, indent=2)
        print(f"Saved results to: {site_result_file}")
        
    except Exception as e:
        print(f"Error processing {file}: {str(e)}")
        continue

# Save combined results
combined_results_file = os.path.join(results_dir, f"all_sites_metrics_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json")
with open(combined_results_file, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f"\n{'='*60}")
print(f"All results saved to: {combined_results_file}")
print(f"{'='*60}")

  0%|          | 0/1 [00:00<?, ?it/s]INFO:p-13230:t-123565036593280:get_model.py:get_model:Loading model from: ibm-granite/granite-timeseries-ttm-r2



Processing: site_113_Shadipur_Delhi_CPCB_15Min.csv
-------------------- Running few-shot 5% --------------------


INFO:p-13230:t-123565036593280:get_model.py:get_model:Model loaded successfully from ibm-granite/granite-timeseries-ttm-r2, revision = main.
INFO:p-13230:t-123565036593280:get_model.py:get_model:[TTM] context_length = 512, prediction_length = 96
INFO:p-13230:t-123565036593280:lr_finder.py:optimal_lr_finder:LR Finder: Running learning rate (LR) finder algorithm. If the suggested LR is very low, we suggest setting the LR manually.
INFO:p-13230:t-123565036593280:lr_finder.py:optimal_lr_finder:LR Finder: Using cuda:0.


Number of params before freezing backbone 805280
Number of params after freezing the backbone 289696


INFO:p-13230:t-123565036593280:lr_finder.py:optimal_lr_finder:LR Finder: Suggested learning rate = 0.0006280291441834253


...

Epoch,Training Loss,Validation Loss
1,0.659900,0.747590
2,0.655500,0.748640
3,0.642600,0.751254
4,0.625400,0.756652
5,0.614700,0.763908
6,0.596900,0.774793
7,0.574900,0.783209
8,0.553200,0.793682
9,0.530600,0.796496
10,0.509800,0.808127


[TrackingCallback] Mean Epoch Time = 0.2649654041637074 seconds, Total Train Time = 11.351677179336548
++++++++++++++++++++ Test MSE after few-shot 5% fine-tuning ++++++++++++++++++++


{'eval_loss': 0.6209022998809814, 'eval_runtime': 0.5434, 'eval_samples_per_second': 9504.634, 'eval_steps_per_second': 149.056, 'epoch': 11.0}
(5165, 96, 6)

MEAN Baseline - PER-CHANNEL METRICS
Shape: 5165 samples × 96 timesteps × 6 channels


100%|██████████| 1/1 [00:18<00:00, 18.02s/it]


MEDIAN Baseline - PER-CHANNEL METRICS
Shape: 5165 samples × 96 timesteps × 6 channels
Saved results to: ttm_benchmarking_results3/site_113_Shadipur_Delhi_CPCB_15Min_metrics.json

All results saved to: ttm_benchmarking_results3/all_sites_metrics_20260222_120805.json
